# Regime A calibration (tuning)

Turn the `WorkloadConfig` knobs until `generate()` output passes the frozen
`spec/regime_A.json` tolerances. `validate()` returns both pass/fail **and** the
MSM objective **Q** to minimize.

Workflow: edit the **Tweak** cell -> re-run it -> watch Q drop -> freeze when seeds pass.

In [ ]:
import dataclasses
from pathlib import Path
import numpy as np
import workload_gen_pipeline as wg

# Robust to cwd: locate the frozen spec next to the package, not via a relative path.
SPEC = Path(wg.__file__).parent / "spec" / "regime_A.json"

def evaluate(cfg, seeds=range(8), show=True):
    """Generate + validate over several seeds. Reports per-check pass-rate with
    current mean vs spec target, plus TWO objectives:
      Q_feasible  -> distance OUTSIDE the tolerances; 0 when all pass. MINIMIZE THIS.
      Q_faithful  -> distance to the real-day point targets (how realistic the trace is).
    Judge a setting on the pass-rate across seeds, NEVER one lucky draw (seed 0).
    Returns (mean Q_feasible, reports)."""
    reports = [wg.validate(wg.generate(cfg, seed=s), SPEC) for s in seeds]
    Qf = [r.distance_feasible for r in reports]
    Qt = [r.distance_faithful for r in reports]
    n = len(reports)
    if show:
        print(f"per-check over {n} seeds   (current mean vs spec target):")
        for i, c in enumerate(reports[0].checks):
            rate = sum(r.checks[i].passed for r in reports)
            cur = np.mean([r.checks[i].value for r in reports])
            flag = "[pass]" if rate == n else "[fail]"
            extra = ""
            if c.bias is not None:  # per_rack_mean_dev: split bias (systematic) vs spread (scatter)
                bias = np.mean([r.checks[i].bias for r in reports])
                spread = np.mean([r.checks[i].spread for r in reports])
                extra = f"  (bias {bias:+.1f}% +/- {spread:.1f}% sd)"
            print(f"  {flag} {c.name:<22} cur {cur:< 11.4g} target {c.target:< 11.4g}  {rate}/{n}{extra}")
        n_all = sum(r.passed for r in reports)
        print(f"mean Q_feasible = {np.mean(Qf):.4g}   (search objective: -> 0 when all pass)")
        print(f"mean Q_faithful = {np.mean(Qt):.4g}   (distance to real-day point targets)")
        print(f"all-checks-pass: {n_all}/{n}")
    return float(np.mean(Qf)), reports

In [54]:
# Baseline: the constraint-derived starting theta
cfg = wg.WorkloadConfig.regime_A_starting(SPEC)
evaluate(cfg);

per-check over 8 seeds   (current mean vs spec target):
  [fail] per_rack_mean_dev      cur  23.91      target  0           0/8  (bias -9.0% +/- 2.5% sd)
  [fail] total_mean_W           cur  1.468e+07  target  1.612e+07   1/8
  [fail] pc1_var_share          cur  0.971      target  0.9942      0/8
  [pass] ramp_excess_kurtosis   cur  186.5      target  25.18       8/8
  [fail] offdiag_corr_mean      cur  0.9664     target  0.9939      6/8
mean Q_feasible = 0.07044   (search objective: -> 0 when all pass)
mean Q_faithful = 43.49   (distance to real-day point targets)
all-checks-pass: 0/8


### Tuning
- **Keep seeds fixed** while turning one knob, so Q changes only from your edit (common random numbers).
- A setting is "passing" only if it passes across **multiple seeds**.
- Read the **dominant squared term** in Q to choose the next knob.
- Knobs **interact** (occupancy moves both total_mean and busy-fraction) -- adjust one at a time.

In [55]:
# === TWEAK CELL: edit knobs, re-run, watch Q ===
cfg = wg.WorkloadConfig.regime_A_starting(SPEC)        # start fresh (comment out to keep tuning the same cfg)

cfg = dataclasses.replace(                             # replace() rebuilds -> re-runs validation
    cfg,
    noise_amp_W = 30_000.0,
    # arrival_rate_per_s = 3.0e-4,
    # job_power = wg.DistSpec("normal", {"mean": 756_000, "std": 50_000}),
    # duration  = wg.DistSpec("lognormal", {"mu": 7.62, "sigma": 1.0}),
    # job_size  = wg.DistSpec("beta", {"a": 30, "b": 1.5}),
    # placement = "contiguous",

    job_size  = wg.DistSpec("beta", {"a": 200, "b": 1})
)
evaluate(cfg);

per-check over 8 seeds   (current mean vs spec target):
  [fail] per_rack_mean_dev      cur  12.16      target  0           3/8  (bias -3.9% +/- 0.2% sd)
  [fail] total_mean_W           cur  1.549e+07  target  1.612e+07   3/8
  [pass] pc1_var_share          cur  0.996      target  0.9942      8/8
  [pass] ramp_excess_kurtosis   cur  75.6       target  25.18       8/8
  [pass] offdiag_corr_mean      cur  0.9957     target  0.9939      8/8
mean Q_feasible = 0.01854   (search objective: -> 0 when all pass)
mean Q_faithful = 4.067   (distance to real-day point targets)
all-checks-pass: 3/8


In [56]:
# Freeze the calibrated config once seeds pass
out = Path(wg.__file__).parent / "spec" / "regime_A_calib.json"
cfg.to_json(out)
print("saved", out)

saved c:\Users\frank\NVAITC_files\NVAITC\optimal_dc\workload_gen\spec\regime_A_calib.json
